# 09b — Bounded queue pressure

Occupancy and offered, accepted, processed, and drained rates are separate bounded-channel observations. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import resolve_analysis_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

batch,_=resolve_analysis_batch('e-backpressure',os.environ.get('E_BACKPRESSURE_DIR'))
artifacts=[] if batch is None else passed_json(batch,'backpressure.json')
if artifacts:
    records=[]
    for path, value in artifacts:
        records.append({
            'condition':path.parent.parent.name,
            'run':path.parent.name,
            'classification':value['classification'],
            'peak_occupancy':value['peak_occupancy'],
            'offered_msg_s':value['rates_msg_s']['offered'],
            'accepted_msg_s':value['rates_msg_s']['accepted'],
            'processed_msg_s':value['rates_msg_s']['processed'],
            'drained_msg_s':value['rates_msg_s']['drained'],
            'rss_within_limit':value['memory']['within_limit'],
        })
    raw=pd.DataFrame(records)
    df=(raw.groupby('condition')
        .agg(N_runs=('run','nunique'), classification=('classification','first'),
             median_peak_occupancy=('peak_occupancy','median'),
             median_offered_msg_s=('offered_msg_s','median'),
             median_accepted_msg_s=('accepted_msg_s','median'),
             median_processed_msg_s=('processed_msg_s','median'),
             median_drained_msg_s=('drained_msg_s','median'),
             rss_within_limit=('rss_within_limit','all'))
        .reset_index())
    df['units']='occupancy ratio, messages/second, boolean'
    df['uncertainty']='descriptive only'
    df['thesis_evidence']=False
    print(f"{evidence_label(len(raw), 'occupancy ratio, messages/second, boolean', False)}; independent runs={len(raw)}")
    display(df)
else:
    display(pd.DataFrame([pending_record('bounded queue pressure','no passed backpressure.json leaf','occupancy ratio, messages/second, boolean')]))
